# Tau Decay Analysis for jGCaMP7s Data

**Purpose:** Determine the optimal tau (calcium decay time constant) for spike inference in Suite2p datasets.

**jGCaMP7s expected tau:** ~1.0s (Dana et al. 2019 Nature Methods)

**Testing:** Compare tau values of 0.8, 1.0, 1.3, and 1.5 seconds

In [ ]:
from pathlib import Path
import lbm_suite2p_python as lsp
import numpy as np
import matplotlib.pyplot as plt

from suite2p.extraction import dcnv

plt.style.use('dark_background')
%matplotlib inline

## Configure Input Path

Set the path to your Suite2p output directory (containing merged/ or plane0/ subdirectories)

In [ ]:
# Configure your Suite2p output path here
suite2p_base_path = Path(r"\\rbo-s1\S1_DATA\lbm\kbarber\2025-11-04-mk311\suite2p")

# Choose which subdirectory to load from (merged, plane0, etc.)
results_subdir = "merged"  # or "plane0", "plane1", etc.

# Optional: specify output directory for figures (defaults to suite2p_base_path.parent / 'decay_time')
output_dir = None  # or Path(r"your\output\path")

print(f"Suite2p base path: {suite2p_base_path}")
print(f"Loading from: {results_subdir}")

## Load Suite2p Results

Load Suite2p output files from the configured path

In [ ]:
# Construct full path to results
results_path = suite2p_base_path / results_subdir

results = lsp.load_planar_results(results_path / "ops.npy")

In [ ]:


# Load Suite2p outputs
ops = np.load(results_path / "ops.npy", allow_pickle=True).item()
F = np.load(results_path / "F.npy")
Fneu = np.load(results_path / "Fneu.npy")
spks_original = np.load(results_path / "spks.npy")
iscell = np.load(results_path / "iscell.npy")

# Get accepted cells only
accepted = iscell[:, 0] == 1
F = F[accepted]
Fneu = Fneu[accepted]
spks_original = spks_original[accepted]

# Extract parameters from ops
fs = ops.get('fs', 30.0)
neucoeff = ops.get('neucoeff', 0.7)
tau_original = ops.get('tau', 1.3)

# Set output directory for figures
if output_dir is None:
    output_dir = suite2p_base_path.parent / 'decay_time'
output_dir.mkdir(exist_ok=True, parents=True)

print(f"Loaded {F.shape[0]} cells, {F.shape[1]} frames")
print(f"Frame rate: {fs:.2f} Hz")
print(f"Original tau: {tau_original}s")
print(f"Duration: {F.shape[1] / fs / 60:.1f} minutes")
print(f"Output directory: {output_dir}")

## Neuropil-Corrected Traces

Suite2p deconvolution operates on neuropil-corrected fluorescence

In [ ]:
F_corrected = F - neucoeff * Fneu
print(f"Neuropil correction: F - {neucoeff} * Fneu")

## Run Spike Inference with Different Tau Values

Testing tau = 0.8, 1.0, 1.3, 1.5 seconds

In [ ]:
tau_values = [0.8, 1.0, 1.3, 1.5]

spks_dict = {}

for tau in tau_values:
    print(f"\nRunning deconvolution with tau={tau}s...")

    spks = dcnv.oasis(
        F=F_corrected,
        batch_size=ops.get('batch_size', 500),
        tau=tau,
        fs=fs,
    )

    spks_dict[tau] = spks

    n_spikes = np.sum(spks > 0)
    spikes_per_cell = n_spikes / F.shape[0]
    print(f"  Total spikes: {n_spikes:,}")
    print(f"  Spikes/cell: {spikes_per_cell:.1f}")
    print(f"  Mean spike amplitude: {np.mean(spks[spks > 0]):.3f}")

print("\nDeconvolution complete for all tau values")

## Spike Count Distributions

Smaller tau = more sensitive = more spikes detected  
Larger tau = less sensitive = fewer spikes detected

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='black')
axes = axes.flatten()

for ax, tau in zip(axes, tau_values):
    spks = spks_dict[tau]
    spikes_per_cell = np.sum(spks > 0, axis=1)

    ax.hist(spikes_per_cell, bins=50, color='cyan', alpha=0.7, edgecolor='white')
    ax.set_xlabel('Spikes per cell', fontweight='bold')
    ax.set_ylabel('Count', fontweight='bold')
    ax.set_title(f'tau = {tau}s', fontweight='bold', fontsize=14)
    ax.grid(alpha=0.3)

    median_spks = np.median(spikes_per_cell)
    mean_spks = np.mean(spikes_per_cell)
    ax.text(0.98, 0.97,
            f'Median: {median_spks:.0f}\nMean: {mean_spks:.1f}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=11, color='lime',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

plt.tight_layout()
plt.savefig(output_dir / 'tau_comparison_spike_counts.png',
            dpi=150, bbox_inches='tight', facecolor='black')
plt.show()

## Example Cell Traces

Pick a highly active cell and compare spike inference across tau values

In [ ]:
# Find most active cell based on fluorescence variance (unbiased)
F_activity = np.std(F_corrected, axis=1)
most_active_idx = np.argmax(F_activity)

print(f"Plotting cell {most_active_idx} (highest F variance)")

# Time window (first 60 seconds)
time_window = 60
n_frames = int(time_window * fs)
time = np.arange(n_frames) / fs

F_cell = F_corrected[most_active_idx, :n_frames]

# Find global max spike amplitude for consistent y-axis
max_spike_amp = max(spks_dict[tau][most_active_idx, :n_frames].max() for tau in tau_values)

# Plot
fig, axes = plt.subplots(len(tau_values) + 1, 1, figsize=(18, 12),
                         facecolor='black', sharex=True)

# Raw fluorescence on top
axes[0].plot(time, F_cell, linewidth=0.8, color='white', alpha=0.9)
axes[0].set_ylabel('F (corrected)', fontweight='bold')
axes[0].set_title(f'Cell {most_active_idx} - Fluorescence and Spike Inference',
                  fontweight='bold', fontsize=14)
axes[0].grid(alpha=0.3)

# Spike inference for each tau with shared y-axis
for i, tau in enumerate(tau_values, start=1):
    spks = spks_dict[tau][most_active_idx, :n_frames]
    spike_times = time[spks > 0]
    spike_amps = spks[spks > 0]

    axes[i].stem(spike_times, spike_amps, linefmt='lime', markerfmt='o', basefmt=' ')
    axes[i].set_ylabel(f'tau={tau}s', fontweight='bold')
    axes[i].set_ylim(0, max_spike_amp * 1.05)
    axes[i].grid(alpha=0.3)

    n_spikes_window = np.sum(spks > 0)
    axes[i].text(0.02, 0.95, f'{n_spikes_window} spikes',
                transform=axes[i].transAxes, va='top', fontsize=10,
                color='cyan', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

axes[-1].set_xlabel('Time (s)', fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / 'tau_comparison_example_cell.png',
            dpi=150, bbox_inches='tight', facecolor='black')
plt.show()

## Spike Amplitude Distributions

How does tau affect the inferred spike amplitudes?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='black')
axes = axes.flatten()

for ax, tau in zip(axes, tau_values):
    spks = spks_dict[tau]
    spike_amps = spks[spks > 0]

    ax.hist(spike_amps, bins=100, range=(0, np.percentile(spike_amps, 99)),
            color='lime', alpha=0.7, edgecolor='white')
    ax.set_xlabel('Spike amplitude', fontweight='bold')
    ax.set_ylabel('Count', fontweight='bold')
    ax.set_title(f'tau = {tau}s', fontweight='bold', fontsize=14)
    ax.set_yscale('log')
    ax.grid(alpha=0.3)

    median_amp = np.median(spike_amps)
    mean_amp = np.mean(spike_amps)
    ax.text(0.98, 0.97,
            f'Median: {median_amp:.3f}\nMean: {mean_amp:.3f}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=11, color='cyan',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

plt.tight_layout()
plt.savefig(output_dir / 'tau_comparison_spike_amplitudes.png',
            dpi=150, bbox_inches='tight', facecolor='black')
plt.show()

## Summary Statistics

Compare all tau values quantitatively

In [ ]:
print("\n" + "="*80)
print("TAU COMPARISON SUMMARY (jGCaMP7s)")
print("="*80)
print(f"{'Tau (s)':<12} {'Total Spikes':<15} {'Spikes/Cell':<15} {'Mean Amp':<12} {'Median Amp':<12}")
print("-"*80)

for tau in tau_values:
    spks = spks_dict[tau]
    total_spikes = np.sum(spks > 0)
    spikes_per_cell = total_spikes / F.shape[0]

    spike_amps = spks[spks > 0]
    mean_amp = np.mean(spike_amps)
    median_amp = np.median(spike_amps)

    marker = "  <-- jGCaMP7s" if tau == 1.0 else ""
    marker = "  <-- Suite2p default" if tau == 1.3 else marker

    print(f"{tau:<12.1f} {total_spikes:<15,} {spikes_per_cell:<15.1f} {mean_amp:<12.4f} {median_amp:<12.4f}{marker}")

print("="*80)
print("\nRecommendation: Use tau=1.0s for jGCaMP7s (Dana et al. 2019)")
print("Default Suite2p tau=1.3s was optimized for GCaMP6s/GCaMP6f")

## Spike Train Correlation

How similar are the spike trains between different tau values?

In [ ]:
correlation_matrix = np.zeros((len(tau_values), len(tau_values)))

for i, tau_i in enumerate(tau_values):
    for j, tau_j in enumerate(tau_values):
        spks_i = (spks_dict[tau_i] > 0).astype(float).flatten()
        spks_j = (spks_dict[tau_j] > 0).astype(float).flatten()
        correlation_matrix[i, j] = np.corrcoef(spks_i, spks_j)[0, 1]

fig, ax = plt.subplots(figsize=(8, 7), facecolor='black')
im = ax.imshow(correlation_matrix, cmap='RdYlGn', vmin=0.8, vmax=1.0)

ax.set_xticks(range(len(tau_values)))
ax.set_yticks(range(len(tau_values)))
ax.set_xticklabels([f'{t}s' for t in tau_values])
ax.set_yticklabels([f'{t}s' for t in tau_values])
ax.set_xlabel('Tau (s)', fontweight='bold', fontsize=12)
ax.set_ylabel('Tau (s)', fontweight='bold', fontsize=12)
ax.set_title('Spike Train Correlation (Binarized)', fontweight='bold', fontsize=14)

for i in range(len(tau_values)):
    for j in range(len(tau_values)):
        text = ax.text(j, i, f'{correlation_matrix[i, j]:.3f}',
                      ha='center', va='center', color='black', fontweight='bold')

plt.colorbar(im, ax=ax, label='Correlation')
plt.tight_layout()
plt.savefig(output_dir / 'tau_spike_correlation.png',
            dpi=150, bbox_inches='tight', facecolor='black')
plt.show()

print("\nCorrelation matrix:")
print(correlation_matrix)

## Conclusion

For jGCaMP7s datasets, tau=1.0s provides the most accurate spike inference based on the indicator's known kinetics (Dana et al. 2019 Nature Methods).

To use in future runs, add to ops: `{'tau': 1.0}`